# Unsupervised Learning

## Introduction

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster import hierarchy

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from itables import show
from pprint import pprint
import os

import folium
import geopandas

from sklearn import decomposition, preprocessing
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS, TSNE
from sklearn.ensemble import IsolationForest

from sentence_transformers import SentenceTransformer

from ind5003 import clust

### Example: Wine quality data

In [ ]:
wine_red = pd.read_csv("data/wine+quality/winequality-red.csv", 
                       delimiter=";" )
wine_red['type'] = "red"
wine_white = pd.read_csv("data/wine+quality/winequality-white.csv", 
                         delimiter=";")
wine_white['type'] = "white"

# remove spaces in column names:
col_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
             'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide',
             'density', 'pH', 'sulphates', 'alcohol', 'quality', 'type']
wine2 = pd.concat([wine_red, wine_white], ignore_index=True)
wine2.columns = col_names

In [ ]:
show(wine2.head(20))

In [ ]:
print(wine2.head())

## Principal Components Analysis
### Formal Set-up
### Example: PCA on wine dataset

In [ ]:
X_raw = wine2.iloc[:, :-2]
scaler = preprocessing.StandardScaler().fit(X_raw)
X_scaled = scaler.transform(X_raw)

In [ ]:
pca_full = decomposition.PCA(n_components=11)
pca_full.fit(X_scaled);

In [ ]:
#| fig-cap: Scree plot for wine principal components
#| fig-pos: ht
#| label: fig-wine-scree

PC_values = np.arange(pca_full.n_components_) + 1
plt.figure(figsize=(6,3))
plt.plot(PC_values, pca_full.explained_variance_ratio_, 'o-', linewidth=2,
	color='blue')
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained');

In [ ]:
pca_full.explained_variance_ratio_.cumsum()

In [ ]:
pca = decomposition.PCA(n_components=5)
pca.fit(X_scaled)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_matrix = pd.DataFrame(loadings, 
                              columns=['PC' + str(x+1) for x in range(0, 5)], 
                              index=col_names[:-2])
loading_matrix2 = loading_matrix.copy()
loading_matrix2[loading_matrix.abs()  < 0.300] = 0.00

In [ ]:
#| echo: false
#| tbl-cap: Loading matrix, wine dataset
#| label: tbl-wine-loading2
loading_matrix2.round(3).style.background_gradient(cmap='coolwarm_r', 
                                                   vmin=-1, vmax=1)

In [ ]:
#| echo: false
#| tbl-cap: Loading matrix, wine dataset
#| label: tbl-wine-loading
loading_matrix2.round(2)

In [ ]:
#| fig-align: center
#| fig-pos: 'ht'
#| fig-cap: Plot of principal components, split by wine rating
#| label: fig-pc-rating

X_transformed = pca.transform(X_scaled)
X_transformed_df = pd.DataFrame(X_transformed, 
                                columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5'])
X_transformed_df[['quality', 'type']] = wine2[['quality', 'type']]

sns.relplot(data=X_transformed_df, x='PC1', y='PC2', col='quality', col_wrap= 3, 
            hue='type', marker='o', alpha=0.3, height=3, aspect=1.2);

## Clustering {#sec-07-clustering}
### Hierarchical Clustering
#### Dissimilarity Measures Between Individual Observations
#### Dissimilarity Measures Between Clusters or Groups
#### Agglomerative Hierarchical Clustering Algorithm

In [ ]:
#| echo: false
X = np.array([[.25,.7], [.3, .8], [.7, .6]])
X

In [ ]:
#| echo: false
pairwise_distances(X).round(3)

In [ ]:
#| fig-align: center
#| fig-cap: "Clustering toy example"
#| label: fig-toy-clustering
#| echo: false
fc_dict={'Stage 0': ['red', 'blue', 'green'], 'Stage 1':['red', 'red', 'green'], 
        'Stage 2':['red']*3}

plt.figure(figsize=(10, 2))
for x,y in enumerate(fc_dict.items()):
    plt.subplot(1,3,x+1);
    plt.scatter(X[:,0], X[:,1], facecolor=y[1]);
    plt.ylim(0.2,1); plt.xlim(0,1);
    plt.title(y[0]);

In [ ]:
#| fig-align: center
#| fig-cap: Dendrogram for toy dataset
#| label: fig-toy-dendrogram
#| fig-pos: 'ht'

lm0 = hierarchy.linkage(X)
plt.figure(figsize=(4, 3))
hierarchy.dendrogram(lm0,p=2)
plt.title('A Dendrogram');

### Example: Hierarchical clustering of wine 

In [ ]:
#| fig-align: center
#| fig-cap: Dendrogram for wine dataset
#| label: fig-wine-dendrogram
#| fig-pos: 'ht'

hc1 = hierarchy.linkage(X_transformed_df.iloc[:, :-2], method='ward')

plt.figure(figsize=(12,4))
hierarchy.dendrogram(hc1, p=4, truncate_mode='level');

### Determining the optimal number of clusters
### Example: Wine clustering quality

In [ ]:
out = hierarchy.cut_tree(hc1, n_clusters=3).ravel() 
X_transformed_df['groups'] = out
clust.compute_silhouette_scores(hc1, X_transformed_df.iloc[:, :-2], [2,3,4])

In [ ]:
#| fig-align: center
#| fig-cap: Wine PC, by red/white wine
#| label: fig-wine-scatter-pc
#| fig-pos: 'ht'

out = hierarchy.cut_tree(hc1, n_clusters=2).ravel() 
X_transformed_df['groups'] = out

sns.relplot(data=X_transformed_df, x='PC1', y='PC2', col='groups', 
            hue='type', marker='o', alpha=0.3, height=3, aspect=1.2);

In [ ]:
#| tbl-cap: Wine counts, by clusters
#| label: tbl-wine-clusters
wine2.type.groupby(X_transformed_df.groups).describe()

## Outlier Detection
### Example: Isolation forest with taiwan dataset

In [ ]:
re2 = pd.read_csv("data/taiwan_dataset.csv")
X_re = re2.loc[:, ['trans_date', 'house_age', 'dist_MRT', 'num_stores', 
                   'Xs', 'Ys', 'price']]
X_re_scaled = preprocessing.StandardScaler().fit_transform(X_re)

In [ ]:
clf = IsolationForest(max_samples=300, max_features=2, contamination=0.01, 
                      random_state=503)
clf.fit(X_re_scaled);

In [ ]:
id_outliers = pd.Series(clf.predict(X_re_scaled))
id_outliers.value_counts()

In [ ]:
re2['outliers'] = id_outliers

In [ ]:
#| tbl-cap: Outlier properties, by explanatory values
#| label: tbl-taiwan-outliers2
show(re2[['house_age', 'dist_MRT', 'num_stores', 'price', 
          'Xs', 'Ys']].groupby(re2.outliers).describe().T)

In [ ]:
#| echo: false
#| tbl-cap: Outlier properties, by explanatory values
#| label: tbl-taiwan-outliers

tmp_df = re2[['house_age', 'dist_MRT', 'num_stores', 'price', 
		'Xs', 'Ys']].groupby(re2.outliers).describe().T
tmp_df.xs("mean", level=1).round(3)

## Visualisation
### MDS
### Example: MDS on disease symptoms

In [ ]:
disease = pd.read_csv("data/disease.csv")

In [ ]:
show(disease)

In [ ]:
disease_names = disease.disease.to_list()
symptoms = disease.columns.to_list()[:-1]

X = disease.iloc[:, 0:-1].to_numpy()

symptom_text = []

for i in range(0, X.shape[0]):
    symptom_text.append(','.join([symptoms[x] for x in np.where(X[i] == 1)[0]]))
disease['symptom_text'] = symptom_text

In [ ]:
disease.loc[disease.disease.isin(['GERD', 
				'Heart attack']), 'symptom_text'].to_list()

In [ ]:
embedding = MDS(n_components=2, normalized_stress='auto', init = 'random',
		        metric='precomputed', n_init=4,
                random_state=42, max_iter=500, verbose=0)

# pdist2 is 41x41
pdist2 = pairwise_distances(X!=0, metric='jaccard')
X_transformed = embedding.fit_transform(pdist2)
X_transformed_df = pd.DataFrame(X_transformed, columns=['X', 'Y'])
X_transformed_df['disease'] = disease_names

In [ ]:
#| fig-align: center
fig = px.scatter(X_transformed_df, x='X', y='Y', text='disease', hover_name=symptom_text,
                 width=900, height=600)
fig.update_traces(textposition='top center')

In [ ]:
#| fig-align: center
#| fig-cap: MDS plot of disease
#| label: fig-mds-disease
#| fig-pos: 'ht'

plt.figure(figsize=(9, 4))
ax = sns.scatterplot(
    data=X_transformed_df, x="X", y="Y"
)

for _, r in X_transformed_df.iterrows():
    ax.text(r["X"], r["Y"], r["disease"],
            ha="center", va="bottom", fontsize=8)

In [ ]:
for x in disease.loc[disease.disease.isin(['Typhoid', 'Malaria']), 'symptom_text'].to_list():
	pprint(x)

### Example: Disease symptoms 3D-plot

In [ ]:
#| fig-align: center
embedding3 = MDS(n_components=3, normalized_stress='auto', metric='precomputed',
                 init = 'random', n_init=4,
                 random_state=42, max_iter=500, verbose=0)
X_transformed3 = embedding3.fit_transform(pdist2)
X_transformed3_df = pd.DataFrame(X_transformed3, columns=['X', 'Y', 'Z'])
X_transformed3_df['disease'] = disease_names

fig = px.scatter_3d(X_transformed3_df, x='X', y='Y', z='Z', text='disease', 
                     width=900, height=600)
fig.show()

### t-SNE {#sec-03-t-SNE}
### Example: Twitter dataset

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
#sentences = ["This is an example sentence.", "Each sentence is converted."]
#embeddings = model.encode(sentences)
#embeddings.shape

In [ ]:
bbchealth_df = pd.read_table('data/health+news+in+twitter/Health-Tweets/bbchealth.txt', 
                             delimiter='|', 
			     names=['id', 'datetime', 'tweet'])
bbchealth_tweets = bbchealth_df.tweet
print(bbchealth_tweets[10])

In [ ]:
t1 = bbchealth_tweets.str.replace(' http.*$', '', regex=True)
t2 = t1.str.replace('^VIDEO:', '', regex=True)
t2_l = t2.to_list()

embeddings = model.encode(t2_l)

### Example: Twitter dataset t-SNE output

In [ ]:
tsne1 = TSNE(n_components=2, init="random", perplexity=10, verbose=0, 
             random_state=43, max_iter=5000)
X_transformed2 = tsne1.fit_transform(embeddings)

df2 = pd.DataFrame(X_transformed2, columns=['x','y'])
df2['labels'] = t2_l

In [ ]:
#| fig-align: center
#| fig-cap: t-SNE plot of Tweets
#| label: fig-tsne-tweets
#| fig-pos: 'ht'
plt.figure(figsize=(8, 4))
sns.scatterplot( data=df2, x="x", y="y")

In [ ]:
#| fig-align: center

fig = px.scatter(df2, x='x', y='y', hover_name='labels',
                 width=900, height=600)
fig.show()

## References
### Website references
### Video references
## Exercises